# Kenya questions: from the students' sheets to the test set

Run this notebook to turn the students' spreadsheets into the Kenya questions the model is tested
on, and **see every step**: which file is which student, every change the clean-up makes, what
gets imported, and the final checks. It runs on the laptop (no GPU) in about a minute.

**How to run:** top right **Select Kernel → `bluff-detector (.venv)`**, then **Run All**.
Re-run it whenever a student sends an updated sheet (after updating `data/kenya/student_files.yaml`).

| Step | What happens | Code |
|---|---|---|
| 1 | Which file is which student | `data/kenya/student_files.yaml` |
| 2 | Clean the sheets; **every change is listed** | `src/bluff/sheet_clean.py` |
| 3 | Look at the changes | `data/kenya/cleaned/changes.csv` |
| 4 | Import into the question file | `src/bluff/sheet_import.py` |
| 5 | Check: data contract and leakage audit | `src/bluff/pipeline/validate.py`, `audit.py` |
| 6 | The final questions | `data/kenya/questions.jsonl` |

The full explanation: `docs/kenya-questions.md`. What each student still has to fix:
`..\Kenya questions - fixes per student.md`.

In [1]:
import os
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display

from bluff import sheet_clean, sheet_import
from bluff.config import load_config
from bluff.pipeline import audit, ingest, validate
from bluff.utils import read_jsonl

if Path.cwd().name == "notebooks":  # work from the project folder
    os.chdir("..")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 60)
cfg = load_config("configs/variants/m0_base.yaml")

## 1. Which file is which student

From `data/kenya/student_files.yaml`, matched to the Student Brief's letters and domains. Edit
that file (not this notebook) when a student sends a new sheet or a name becomes known. Rows
listed under `exclude` are left out on purpose, with the reason.

In [2]:
specs = yaml.safe_load(Path("data/kenya/student_files.yaml").read_text(encoding="utf-8"))
display(pd.DataFrame([{
    "student": s["student"], "name": s["author"], "domain": s["domain"],
    "file": s["file"], "tab": s["tab"], "file found": Path(s["file"]).exists(),
    "rows excluded": len(s.get("exclude") or {}),
} for s in specs]).set_index("student"))

,name,domain,file,tab,file found,rows excluded
student,,,,,,
A,James Gosha,civics_law,../Bluff Detector - Kenya Question Sheet student b.xlsx,Student B,True,3
B,Student B,geography_society,../Geography_Questions.xlsx,Sheet1,True,0
C,Michael,institutions_services,../Bluff Detector - Kenya Question Sheet 1.xlsx,Student C,True,0
D,Student D,western_default_traps,../project 2.xlsx,Student D,True,0


## 2. Clean the sheets

Only mechanical fixes that don't change what a question asks: add a missing "?", add
"In Kenya, " when the question never names the country, show Excel-damaged answers as typed
(16% stored as 0.16, dates), map the domain onto the brief's six, split "999/112" answers, fill a
missing Author, and skip rows that can't be used (empty, headings, no answer, about another
country, repeats). Writes `data/kenya/cleaned/cleaned_questions.xlsx` and the change list.

In [3]:
result = sheet_clean.run(cfg)
counts = pd.DataFrame(result["students"]).fillna(0).astype(int)
print(f"{result['changes']} changes recorded")
display(counts)

490 changes recorded


,Student A (James Gosha),Student B (Student B),Student C (Michael),Student D (Student D)
"added 'In Kenya, '",16,11,17,3
domain mapped to the brief's six,45,83,86,82
kept: real,71,73,66,72
excluded in student_files.yaml: copies the Constitution's wording (Chapter 7): leakage audit; rewrite in own words,1,0,0,0
excluded in student_files.yaml: copies the Constitution's wording (Chapter 5): leakage audit; rewrite in own words,1,0,0,0
Excel value shown as typed,9,2,1,1
excluded in student_files.yaml: copies the Constitution's wording (Chapter 9): leakage audit; rewrite in own words,1,0,0,0
kept: fake,20,20,20,10
skipped: no question text,15,0,0,1
Author filled from student_files.yaml,0,93,0,82


## 3. Every change, row by row

The same list is in the **Changes** tab of `cleaned_questions.xlsx` and in `changes.csv`. Below:
how many of each kind, then a few examples of each. To see all of one kind, change `KIND`.

In [4]:
changes = pd.read_csv("data/kenya/cleaned/changes.csv", encoding="utf-8-sig")
display(changes.groupby(["why", "student"]).size().unstack(fill_value=0))

student,Student A,Student B,Student C,Student D
why,,,,
Excel value shown as typed,9,2,1,1
added '?',0,0,5,82
"added 'In Kenya, '",16,11,17,3
domain mapped to the brief's six,45,83,86,82
excluded in student_files.yaml: copies the Constitution's wording (Chapter 5): leakage audit; rewrite in own words,1,0,0,0
excluded in student_files.yaml: copies the Constitution's wording (Chapter 7): leakage audit; rewrite in own words,1,0,0,0
excluded in student_files.yaml: copies the Constitution's wording (Chapter 9): leakage audit; rewrite in own words,1,0,0,0
"skipped: about another country, not Kenya",0,0,0,10
skipped: no Type (a heading or unfinished row),0,8,0,1


In [5]:
for why, g in changes.groupby("why"):
    display(Markdown(f"**{why}** ({len(g)})"))
    display(g[["student", "row", "before", "after"]].head(3))

**Excel value shown as typed** (13)

,student,row,before,after
34,Student A,24,2010-08-27 00:00:00,27 August 2010
40,Student A,35,2026-06-01 00:00:00,1 June
41,Student A,36,2026-10-20 00:00:00,20 October


**added '?'** (87)

,student,row,before,after
257,Student C,57,What is Kenya's leading foreign-exchange-earning agricultural export,What is Kenya's leading foreign-exchange-earning agricultural export?
259,Student C,58,Which regulator licenses the NSE,Which regulator licenses the NSE?
274,Student C,66,Which Act of Parliament established the capital markets authority,Which Act of Parliament established the capital markets authority?


**added 'In Kenya, '** (47)

,student,row,before,after
0,Student A,2,Who is incharge of the registration of candidates for election?,"In Kenya, who is incharge of the registration of candidates for election?"
2,Student A,3,Who is in charge of the settlement of electoral disputes?,"In Kenya, who is in charge of the settlement of electoral disputes?"
4,Student A,4,Who is in charge of voter education?,"In Kenya, who is in charge of voter education?"


... (16 outputs left out)

**skipped: repeats an earlier question or a starter** (4)

,student,row,before,after
91,Student B,14,What is the capital city of Kenya?,NaN
237,Student C,42,Which department issues Kenyan passports?,NaN
256,Student C,56,Which agency certifies horticultural exports for phytosanitary compliance?,NaN


**split '/' or 'or' answer into accepted answers** (3)

,student,row,before,after
402,Student D,49,NaN,National government; County government
407,Student D,51,Mkuki,Mkuki; Shield; Spears
413,Student D,53,swahili/kiingeereza,swahili/kiingeereza; English; Kiswahili


In [6]:
KIND = "added 'In Kenya, '"  # any value from the 'why' column above
changes[changes["why"] == KIND][["student", "row", "before", "after"]]

,student,row,before,after
0,Student A,2,Who is incharge of the registration of candidates for election?,"In Kenya, who is incharge of the registration of candidates for election?"
2,Student A,3,Who is in charge of the settlement of electoral disputes?,"In Kenya, who is in charge of the settlement of electoral disputes?"
4,Student A,4,Who is in charge of voter education?,"In Kenya, who is in charge of voter education?"
6,Student A,5,Who is in charge of the delimitation of constituencies and wards?,"In Kenya, who is in charge of the delimitation of constituencies and wards?"
8,Student A,6,"Who is in charge of facilitating the observation, monitoring and evaluation ...","In Kenya, who is in charge of facilitating the observation, monitoring and e..."
11,Student A,8,Who is in charge of the continuous registration of voters and the revision o...,"In Kenya, who is in charge of the continuous registration of voters and the ..."
13,Student A,9,Who is in charge of monitoring compliance with the legislation on the nomina...,"In Kenya, who is in charge of monitoring compliance with the legislation on ..."
15,Student A,10,who is the administrative head of the national assembly?,"In Kenya, who is the administrative head of the national assembly?"
17,Student A,11,who is incharge of management of parliamentary internship programmes?,"In Kenya, who is incharge of management of parliamentary internship programmes?"
19,Student A,12,who maintains custody of the mace in parliament?,"In Kenya, who maintains custody of the mace in parliament?"


## 4. Import into the question file

Each cleaned row becomes one question in `data/kenya/questions.jsonl` (real → `kenya`, fake →
`kenya_unanswerable`, twin → `western`), with an id made from its text, so re-importing keeps
every earlier id and the GPU run only asks new or changed questions. Trevor's 9 starter
questions are kept.

In [7]:
imported = sheet_import.run(cfg, "data/kenya/cleaned/cleaned_questions.xlsx")
print(f"{imported['imported']} imported from the students, {imported['starters_kept']} starters "
      f"kept: {imported['total']} Kenya questions (was {imported['previous_total']})")
display(pd.DataFrame(imported["by_author"]).T)
display(Markdown(Path(imported["report"]).read_text(encoding="utf-8")))

352 imported from the students, 9 starters kept: 361 Kenya questions (was 361)


,kenya,kenya_unanswerable,western,verified
Trevor,7,2,0,0
JAMES GOSHA,71,20,0,0
Student B,73,20,0,0
Michael,66,20,0,0
Student D,72,10,0,0


# Kenya question import report

Imported 2026-11-19 12:45 from `cleaned_questions.xlsx`: 352 filled rows; the question file now holds 361 questions (it held 361).

## Questions per author

| Author | Real | Fake | Twin | Verified |
|---|---|---|---|---|
| JAMES GOSHA | 71 | 20 | 0 | 0 |
| Michael | 66 | 20 | 0 | 0 |
| Student B | 73 | 20 | 0 | 0 |
| Student D | 72 | 10 | 0 | 0 |
| Trevor | 7 | 2 | 0 | 0 |

## Rows with problems (0)

None.


## 5. Checks

**Data contract** (`validate`): ids unique, real questions have answers, fakes have none, pools
correct. **Leakage audit** (`audit`): no question's wording may appear in the documents the RAG
model reads or the fine-tuned model is trained on; a question that copies a source would make
the test unfair. The audit needs the full question list, so `ingest` runs first (it uses the
cached TriviaQA download).

In [8]:
print("data contract:", validate.run(cfg))
m1 = load_config("configs/variants/m1_rag.yaml")
ingest.run(m1)
try:
    print("leakage audit:", audit.run(m1))
except ValueError as e:
    print("LEAKAGE AUDIT FAILED:", e)
    print("See data/processed/leakage_audit.json; exclude the rows in student_files.yaml.")

361 of 361 Kenya items are not yet verified by a second reviewer


data contract: {'n': 361, 'answerable': 289, 'unanswerable': 72, 'verified': 0}


C:\Users\Admin\OneDrive\Desktop\Bluff_Detector\bluff-detector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


leakage audit: {'retrieval_corpus': {'checked': True, 'overlaps': 0, 'examples': []}, 'finetune_corpus': {'checked': True, 'overlaps': 0, 'examples': []}}


## 6. The final Kenya questions

What the model will be asked, per student and type, and a few examples of each. Next: upload to
GitHub (`git add data/kenya`, `git commit`, `git push`), then run `notebooks/run_m0_base.ipynb`
on Kaggle or Colab: it asks only the new questions.

In [9]:
q = pd.DataFrame(read_jsonl(cfg["data"]["kenya_questions"]))
display(pd.crosstab(q["author"], q["pool"], margins=True, margins_name="total"))
examples = pd.concat([d.sample(min(4, len(d)), random_state=1) for _, d in q.groupby("pool")])
display(examples[["id", "pool", "author", "question", "gold"]])

pool,kenya,kenya_unanswerable,total
author,,,
JAMES GOSHA,71,20,91
Michael,66,20,86
Student B,73,20,93
Student D,72,10,82
Trevor,7,2,9
total,289,72,361


,id,pool,author,question,gold
154,ke-sw-ccddbfa2,kenya,Student B,What is the most common Swahili greeting?,Habari yako
327,ke-trap-1567243f,kenya,Student D,"In Kenya, what is the main exam that that highschool students do?",KCSE
292,ke-trap-c135242b,kenya,Student D,who was the first president of Kenya?,Jomo kenyatta
64,ke-ins-f0540efe,kenya,JAMES GOSHA,Which authority regulates road transport safety and issues driving licences ...,National Transport and Safety Authority
97,ke-una-civ-56501a8e,kenya_unanswerable,JAMES GOSHA,Jina la wimbo wa pili wa taifa wa Kenya ni lipi?,NaN
272,ke-una-ins-8c4387d8,kenya_unanswerable,Michael,Is KRA the State Corporation in Kenya that provides career guidance?,NaN
88,ke-una-civ-a40dcb88,kenya_unanswerable,JAMES GOSHA,Which Kenyan county borders the Atlantic Ocean?,NaN
357,ke-una-trap-87cb7f5d,kenya_unanswerable,Student D,which date is oktoberfest held in Kenya?,NaN
